# **Forward Chaining Inference Engine (Data-Driven Reasoning)**

In [1]:
# Knowledge base: IF conditions THEN conclusion

rules = [
    {
        "id": "R1",
        "conditions": {"has_hair"},
        "conclusion": "is_mammal"
    },
    {
        "id": "R2",
        "conditions": {"gives_milk"},
        "conclusion": "is_mammal"
    },
    {
        "id": "R3",
        "conditions": {"has_feathers"},
        "conclusion": "is_bird"
    },
    {
        "id": "R4",
        "conditions": {"is_mammal", "eats_meat"},
        "conclusion": "is_carnivore"
    },
    {
        "id": "R5",
        "conditions": {"is_mammal", "has_hooves"},
        "conclusion": "is_ungulate"
    },
    {
        "id": "R6",
        "conditions": {
            "is_carnivore",
            "tawny_color",
            "black_stripes"
        },
        "conclusion": "is_tiger"
    },
    {
        "id": "R7",
        "conditions": {
            "is_ungulate",
            "long_neck",
            "long_legs",
            "spots"
        },
        "conclusion": "is_giraffe"
    },
    {
        "id": "R8",
        "conditions": {
            "is_ungulate",
            "black_stripes",
            "not_carnivore"
        },
        "conclusion": "is_zebra"
    }
]


def forward_chaining(initial_facts, rules):
    working_memory = set(initial_facts)
    firing_sequence = []

    while True:
        fired = False

        for rule in rules:
            if (rule["conditions"] <= working_memory
                    and rule["conclusion"] not in working_memory):

                working_memory.add(rule["conclusion"])
                firing_sequence.append(rule["id"])

                print(
                    f'{rule["id"]} fired: '
                    f'{rule["conclusion"]}'
                )

                fired = True
                break  # Fire only the first applicable rule

        if not fired:
            break

    return working_memory, firing_sequence


def identify_animal(facts):
    animals = {
        "is_tiger": "Tiger",
        "is_giraffe": "Giraffe",
        "is_zebra": "Zebra"
    }

    for fact, animal in animals.items():
        if fact in facts:
            return animal

    return None


# Test case 1: Tiger
initial_facts = {
    "has_hair",
    "eats_meat",
    "tawny_color",
    "black_stripes"
}

final_facts, sequence = forward_chaining(initial_facts, rules)

print("\nFinal Working Memory:")
print(sorted(final_facts))

print("\nFiring Sequence:", sequence)
print("Identified Animal:", identify_animal(final_facts))

R1 fired: is_mammal
R4 fired: is_carnivore
R6 fired: is_tiger

Final Working Memory:
['black_stripes', 'eats_meat', 'has_hair', 'is_carnivore', 'is_mammal', 'is_tiger', 'tawny_color']

Firing Sequence: ['R1', 'R4', 'R6']
Identified Animal: Tiger


In [2]:
# Test case 2: Giraffe
giraffe_facts = {
    "has_hair",
    "has_hooves",
    "long_neck",
    "long_legs",
    "spots"
}

final_facts, sequence = forward_chaining(giraffe_facts, rules)

print("\nGiraffe test:")
print("Firing sequence:", sequence)
print("Identified animal:", identify_animal(final_facts))


# Test case 3: Zebra
zebra_facts = {
    "has_hair",
    "has_hooves",
    "black_stripes",
    "not_carnivore"
}

final_facts, sequence = forward_chaining(zebra_facts, rules)

print("\nZebra test:")
print("Firing sequence:", sequence)
print("Identified animal:", identify_animal(final_facts))

R1 fired: is_mammal
R5 fired: is_ungulate
R7 fired: is_giraffe

Giraffe test:
Firing sequence: ['R1', 'R5', 'R7']
Identified animal: Giraffe
R1 fired: is_mammal
R5 fired: is_ungulate
R8 fired: is_zebra

Zebra test:
Firing sequence: ['R1', 'R5', 'R8']
Identified animal: Zebra


In [3]:
def identify_most_specific_class(facts):
    # Check specific animal identifications first
    if "is_tiger" in facts:
        return "Tiger"

    if "is_giraffe" in facts:
        return "Giraffe"

    if "is_zebra" in facts:
        return "Zebra"

    # Otherwise, report the most specific general class
    if "is_carnivore" in facts:
        return "Carnivore"

    if "is_ungulate" in facts:
        return "Ungulate"

    if "is_mammal" in facts:
        return "Mammal"

    if "is_bird" in facts:
        return "Bird"

    return "Unknown class"


# Facts that establish a class but do not identify an animal
initial_facts = {"has_hair"}

final_facts, sequence = forward_chaining(initial_facts, rules)

print("\nFinal Working Memory:", sorted(final_facts))
print("Firing Sequence:", sequence)

animal = identify_animal(final_facts)

if animal:
    print("Identified animal:", animal)
else:
    print("No specific animal identified.")
    print("Most specific class:", identify_most_specific_class(final_facts))

R1 fired: is_mammal

Final Working Memory: ['has_hair', 'is_mammal']
Firing Sequence: ['R1']
No specific animal identified.
Most specific class: Mammal


In [4]:
def forward_chaining_all(initial_facts, rules):
    working_memory = set(initial_facts)
    firing_sequence = []

    while True:
        # Collect all rules applicable at the start of this cycle
        conflict_set = [
            rule for rule in rules
            if rule["conditions"] <= working_memory
            and rule["conclusion"] not in working_memory
        ]

        if not conflict_set:
            break

        # Fire all rules in the conflict set
        new_facts = set()

        for rule in conflict_set:
            new_facts.add(rule["conclusion"])
            firing_sequence.append(rule["id"])

            print(
                f'{rule["id"]} fired: '
                f'{rule["conclusion"]}'
            )

        working_memory.update(new_facts)

    return working_memory, firing_sequence


# Compare using the same tiger observations
tiger_facts = {
    "has_hair",
    "eats_meat",
    "tawny_color",
    "black_stripes"
}

print("=== Original: One rule per cycle ===")
facts_one, sequence_one = forward_chaining(tiger_facts, rules)

print("\nSequence:", sequence_one)

print("\n=== Modified: All applicable rules per cycle ===")
facts_all, sequence_all = forward_chaining_all(tiger_facts, rules)

print("\nSequence:", sequence_all)

print("\nSame final facts:", facts_one == facts_all)

=== Original: One rule per cycle ===
R1 fired: is_mammal
R4 fired: is_carnivore
R6 fired: is_tiger

Sequence: ['R1', 'R4', 'R6']

=== Modified: All applicable rules per cycle ===
R1 fired: is_mammal
R4 fired: is_carnivore
R6 fired: is_tiger

Sequence: ['R1', 'R4', 'R6']

Same final facts: True
